# Sourcing Funnel Analysis with AI-Assisted Candidate Prioritization

This notebook analyzes the sourcing funnel, conversion patterns, candidate signals, rejection/drop-off reasons, and AI-assisted recruiter-note tags.


## 1. Setup


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = 'mock_sourcing_dataset.csv'
df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
df.shape, df.columns.tolist()


## 2. Funnel Analysis


In [ ]:
stages = {
    'Sourced': df.index == df.index,
    'Responded': df['response_received'] == True,
    'Screening Pass': df['screening_pass'] == True,
    'Interview 1 Pass': df['interview1_pass'] == True,
    'Test Taken': df['test_taken'] == True,
    'Offer Sent': df['offer_sent'] == True,
    'Hired': df['hired'] == True,
}
funnel = []
previous = None
for stage, mask in stages.items():
    count = int(mask.sum())
    funnel.append({
        'stage': stage,
        'count': count,
        'conversion_from_total': count / len(df),
        'conversion_from_previous': 1 if previous is None else count / previous
    })
    previous = count
funnel_df = pd.DataFrame(funnel)
funnel_df


In [ ]:
ax = funnel_df.plot(kind='bar', x='stage', y='count', legend=False, figsize=(9, 5), title='Sourcing Funnel Volume')
ax.set_ylabel('Candidates')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


## 3. Conversion by Channel, Recruiter, and Seniority


In [ ]:
def conversion_table(group_col):
    grouped = df.groupby(group_col).agg(
        sourced=('candidate_id', 'count'),
        responded=('response_received', 'sum'),
        screening_pass=('screening_pass', 'sum'),
        interview1_pass=('interview1_pass', 'sum'),
        test_taken=('test_taken', 'sum'),
        offer_sent=('offer_sent', 'sum'),
        hired=('hired', 'sum')
    ).reset_index()
    for col in ['responded', 'screening_pass', 'interview1_pass', 'test_taken', 'offer_sent', 'hired']:
        grouped[f'{col}_rate'] = grouped[col] / grouped['sourced']
    grouped['test_to_offer_rate'] = grouped['offer_sent'] / grouped['test_taken'].replace(0, pd.NA)
    grouped['offer_to_hired_rate'] = grouped['hired'] / grouped['offer_sent'].replace(0, pd.NA)
    return grouped.sort_values('hired_rate', ascending=False)

source_conversion = conversion_table('source_channel')
recruiter_conversion = conversion_table('recruiter')
seniority_conversion = conversion_table('seniority')
source_conversion


In [ ]:
for table, label in [(source_conversion, 'source_channel'), (recruiter_conversion, 'recruiter'), (seniority_conversion, 'seniority')]:
    ax = table.plot(kind='bar', x=label, y='hired_rate', legend=False, figsize=(9, 5), title=f'Hired Rate by {label}')
    ax.set_ylabel('Hired Rate')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()


## 4. Candidate Signals


In [ ]:
signals = ['technical_test_score', 'behavior_score', 'manager_score', 'years_experience', 'response_time_days']
signal_summary = df.groupby('hired')[signals].mean().T
signal_summary.columns = ['Not Hired', 'Hired']
signal_summary


## 5. Rejection and Drop-off Reasons


In [ ]:
rejection_reasons = df['rejection_reason'].value_counts(dropna=True).reset_index()
rejection_reasons.columns = ['rejection_reason', 'count']
rejection_reasons['share'] = rejection_reasons['count'] / rejection_reasons['count'].sum()
rejection_reasons


## 6. AI-Assisted Recruiter Note Tagging

AI was used to define a simple taxonomy for qualitative recruiter notes. For reproducibility, the taxonomy is implemented below as a deterministic tagging function. In a production workflow, this could be replaced by an LLM classifier with human review.


In [ ]:
def tag_note(note):
    note = str(note).lower()
    tags = []
    if any(term in note for term in ['expectativa salarial', 'salário', 'salarial', 'salary']):
        tags.append('Salary risk')
    if any(term in note for term in ['baixo engajamento', 'low engagement']):
        tags.append('Low engagement')
    if any(term in note for term in ['prazo adicional', 'retorno', 'timing', 'disponibilidade']):
        tags.append('Follow-up/timing risk')
    if any(term in note for term in ['perfil técnico acima', 'técnico acima', 'experiência sólida', 'strong technical']):
        tags.append('Strong technical fit')
    if any(term in note for term in ['potencial', 'potential']):
        tags.append('Potential')
    return tags if tags else ['General note']

df['ai_note_tags'] = df['recruiter_notes'].apply(tag_note)
tag_counts = df.explode('ai_note_tags')['ai_note_tags'].value_counts().reset_index()
tag_counts.columns = ['ai_note_tag', 'count']
tag_counts


## 7. Recruiter Recommendations

- Prioritize high-converting channels such as GitHub, Inbound, and Hunting for similar roles.
- Investigate the test-to-offer stage, since it is the main bottleneck.
- Persist with candidates showing strong technical and manager scores, especially when the concern is timing rather than capability.
- Deprioritize candidates with repeated no-response, low engagement, or clear salary mismatch unless the role has flexibility.
- Use AI note tags to flag risks earlier and standardize recruiter follow-up decisions.
